# Collections Recommendation Specialist — Colab QLoRA (Week 2)

Trains a LoRA adapter on the **pseudonymized** train split (15 examples), evaluates the **holdout** (10 examples), and packages the adapter for download.

**Privacy:** use only repo gold JSONL. Do not upload real customer PII.

**Setup:** `Runtime → Change runtime type → GPU` (T4 or better), then **Run all**.

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("cuda:", torch.cuda.get_device_name(0), "vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

## 2) Install deps (PEFT + TRL + bitsandbytes)

Uses standard Hugging Face stack (more reliable on Colab than Unsloth pin churn).

In [ ]:
%pip install -q -U transformers datasets peft trl bitsandbytes accelerate sentencepiece protobuf

## 3) Get data

Default: clone the public repo (pseudonymized gold only).

Or set `USE_UPLOAD = True` and upload `train_only.jsonl` + `examples.jsonl`.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

USE_UPLOAD = False  # True = use Colab file upload instead of git clone
REPO_DIR = Path("/content/agent")
WORK = Path("/content/collections_lora")
WORK.mkdir(parents=True, exist_ok=True)

if USE_UPLOAD:
    from google.colab import files
    print("Upload train_only.jsonl and examples.jsonl")
    uploaded = files.upload()
    for name in uploaded:
        Path(name).write_bytes(uploaded[name])
    TRAIN_JSONL = Path("train_only.jsonl")
    EXAMPLES = Path("examples.jsonl")
    assert TRAIN_JSONL.exists() and EXAMPLES.exists()
else:
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/selvera247/agent.git", str(REPO_DIR)])
    EXAMPLES = REPO_DIR / "data/collections_specialist/examples.jsonl"
    sys.path.insert(0, str(REPO_DIR / "scripts"))
    # Build train_only in-notebook (works even if prepare script missing on older main)
    SYSTEM = (
        "You are a strict collections recommendation assistant. Use only facts "
        "in the provided JSON input. Output a single JSON object with keys: "
        "action, reason, confidence (0-1), evidence (list of strings). "
        "Do not invent facts. JSON only, no markdown. "
        "action must be one of: Route to dispute specialist | Send automated reminder | "
        "Call AP contact | Email payment reminder | Escalate to manager | "
        "Hold — await promise-to-pay | Reassign to available collector | "
        "Keep parent-child grouping. "
        "If collector availability is pto/unavailable, action must be "
        "Reassign to available collector."
    )
    rows = []
    for line in EXAMPLES.read_text().splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        if (row.get("metadata") or {}).get("split") == "holdout":
            continue
        inp = {k: row[k] for k in ["id","organization_id","customer","invoices","collector_context","policy_flags","instruction"]}
        out = {
            "action": row["expected_action"],
            "reason": row["expected_reason"],
            "confidence": row["expected_confidence"],
            "evidence": row["expected_evidence"],
        }
        rows.append({
            "id": row["id"],
            "messages": [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": "INPUT:\n" + json.dumps(inp, ensure_ascii=False)},
                {"role": "assistant", "content": json.dumps(out, ensure_ascii=False)},
            ],
        })
    TRAIN_JSONL = WORK / "train_only.jsonl"
    TRAIN_JSONL.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in rows) + "\n")
    print("train chats:", len(rows), "->", TRAIN_JSONL)

print("examples:", EXAMPLES)
print("train:", TRAIN_JSONL, "lines:", sum(1 for _ in TRAIN_JSONL.open() if _.strip()))

## 4) Train QLoRA

Defaults match `configs/collections_specialist_config.yaml`: r=32, alpha=64, lr=1e-4, epochs=2.

In [ ]:
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
OUT_DIR = WORK / "adapter"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 32
LORA_ALPHA = 64
LR = 1e-4
EPOCHS = 2
BATCH_SIZE = 1
GRAD_ACCUM = 8
MAX_SEQ = 2048

records = [json.loads(l) for l in TRAIN_JSONL.read_text().splitlines() if l.strip()]

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(example):
    # Chat template → single training string
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

ds = Dataset.from_list(records).map(to_text)
print(ds)
print(ds[0]["text"][:400], "...")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="bfloat16",
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    ),
)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=str(WORK / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    logging_steps=1,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    processing_class=tokenizer,
)
# Older TRL used tokenizer= / dataset_text_field=
try:
    trainer.train()
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=ds,
        tokenizer=tokenizer,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
    )
    trainer.train()

model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
(OUT_DIR / "adapter_ready.json").write_text(json.dumps({
    "base_model": BASE_MODEL,
    "lora_r": LORA_R,
    "epochs": EPOCHS,
    "train_examples": len(records),
}))
print("saved adapter ->", OUT_DIR)

## 5) Holdout inference + score

Generates predictions with the fine-tuned adapter and scores with the repo evaluator when available.

In [ ]:
import re, time, json
from peft import PeftModel

examples = [json.loads(l) for l in Path(EXAMPLES).read_text().splitlines() if l.strip()]
holdout = [e for e in examples if (e.get("metadata") or {}).get("split") == "holdout"]
print("holdout:", len(holdout))

# Reload base+adapter for generate
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype="bfloat16",
)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
ft = PeftModel.from_pretrained(base, str(OUT_DIR))
ft.eval()
tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

SYSTEM = (
    "You are a strict collections recommendation assistant. Use only facts in the provided JSON input. "
    "Output a single JSON object with keys: action, reason, confidence (0-1), evidence (list of strings). "
    "Do not invent facts. JSON only, no markdown."
)

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`")
        if text.startswith("json"):
            text = text[4:].strip()
    try:
        return json.loads(text)
    except Exception:
        a, b = text.find("{"), text.rfind("}")
        return json.loads(text[a:b+1])

preds = []
for ex in holdout:
    payload = {k: ex[k] for k in ["id","organization_id","customer","invoices","collector_context","policy_flags","instruction"]}
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": "INPUT:\n" + json.dumps(payload, ensure_ascii=False) + "\n\nRespond with JSON only."},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(ft.device)
    t0 = time.time()
    out_ids = ft.generate(**inputs, max_new_tokens=256, do_sample=False, temperature=None)
    gen = tok.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    latency = (time.time() - t0) * 1000
    try:
        parsed = extract_json(gen)
    except Exception as e:
        parsed = {"action": "", "reason": gen[:200], "confidence": 0.0, "evidence": [], "parse_error": str(e)}
    preds.append({"id": ex["id"], **parsed, "latency_ms": latency, "raw": gen})
    print(ex["id"], "->", parsed.get("action"), f"({latency:.0f} ms)")

pred_path = WORK / "adapter_predictions.jsonl"
pred_path.write_text("\n".join(json.dumps(p, ensure_ascii=False) for p in preds) + "\n")
print("wrote", pred_path)

In [ ]:
# Score with repo evaluator if present; else print crude action match
metrics = {}
try:
    sys.path.insert(0, str(REPO_DIR / "scripts"))
    from evaluate_collections_recommendations import load_jsonl, score_example, aggregate
    from pathlib import Path as P
    # load_jsonl expects Path
    examples_rows = [json.loads(l) for l in Path(EXAMPLES).read_text().splitlines() if l.strip()]
    hold = [e for e in examples_rows if (e.get("metadata") or {}).get("split") == "holdout"]
    by_id = {p["id"]: p for p in preds}
    scores = [score_example(ex, by_id.get(ex["id"])) for ex in hold]
    agg = aggregate(scores)
    metrics = {k: v for k, v in agg.items() if k != "top_failures"}
    metrics["top_failures"] = [
        {"id": t["id"], "failures": t["failures"], "expected_action": t.get("expected_action"), "predicted_action": t.get("predicted_action")}
        for t in agg["top_failures"]
    ]
except Exception as e:
    print("evaluator import failed, using crude match:", e)
    hold = [e for e in examples if (e.get("metadata") or {}).get("split") == "holdout"]
    by_id = {p["id"]: p for p in preds}
    matches = sum(1 for h in hold if (by_id.get(h["id"]) or {}).get("action") == h["expected_action"])
    metrics = {"n": len(hold), "action_match_rate": matches / max(len(hold), 1), "note": str(e)}

week1 = {
    "action_match_rate": 0.60,
    "policy_pass_rate": 1.00,
    "groundedness_rate": 0.80,
    "hallucination_rate": 0.20,
}
print("Week 1 lock:", week1)
print("Colab adapter:", {k: metrics.get(k) for k in ["action_match_rate","policy_pass_rate","groundedness_rate","hallucination_rate","evidence_coverage_mean","median_latency_ms"]})
metrics_path = WORK / "colab_holdout_metrics.json"
metrics_path.write_text(json.dumps({"week1_lock": week1, "adapter": metrics}, indent=2))
print("wrote", metrics_path)

## 6) Download adapter zip

Unzip on your PC into `C:\Users\Selvera\agent\models\collections-lora\`, then paste metrics into chat / save under `artifacts\collections_eval\`.

In [ ]:
import shutil
from google.colab import files

zip_base = WORK / "collections-lora-adapter"
shutil.make_archive(str(zip_base), "zip", root_dir=OUT_DIR)
# also bundle metrics + predictions next to zip
bundle = WORK / "collections-lora-bundle"
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()
shutil.copytree(OUT_DIR, bundle / "adapter")
shutil.copy(pred_path, bundle / "adapter_predictions.jsonl")
shutil.copy(metrics_path, bundle / "colab_holdout_metrics.json")
shutil.make_archive(str(WORK / "collections-lora-bundle"), "zip", root_dir=bundle)
print("downloading...")
files.download(str(WORK / "collections-lora-bundle.zip"))

### After download (on your Windows PC)

```powershell
cd C:\Users\Selvera\agent
# Expand-Archive collections-lora-bundle.zip -DestinationPath models\collections-lora-bundle
# Copy adapter\* into models\collections-lora\
# Copy colab_holdout_metrics.json to artifacts\collections_eval\
```

Compare `adapter.action_match_rate` etc. to Week 1 lock (0.60 / 1.00 / 0.80 / 0.20).
Paste `colab_holdout_metrics.json` back into the Cursor chat for Week 2 wrap-up.